# Possession Lens — official challenge evaluation

Run the organisers’ unchanged evaluation program locally and view its scoreboard, per-state scores, possession counts and confidence intervals.

**Start:** choose the **Python (Possession Lens)** kernel and select **Run → Run All Cells**. The default example scores the organisers’ published baseline. To evaluate your frozen submission, save its CSV beside this notebook in the `evaluation` folder. The notebook automatically selects a single CSV. You can also type its name in `CSV_FILENAME` below.

The public reference covers only the first **five minutes /17 possessions**. This is a pipeline check, not your final challenge score. Never use it to choose models, tune parameters or manually correct predictions. No model is trained here; no video is opened; no data is uploaded. The full-match reference is private to the organisers.


In [ ]:
from pathlib import Path
import sys, json, hashlib, shutil, subprocess, os
from datetime import datetime
import pandas as pd
from IPython.display import display, FileLink

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "evaluation/official/SOURCE.json").is_file()), None)
assert ROOT is not None, "Open this notebook inside the Possession Lens project."
EVALUATION = ROOT / "evaluation"
OFFICIAL = EVALUATION / "official"
source = json.loads((OFFICIAL / "SOURCE.json").read_text())
for name, record in source["files"].items():
    asset = OFFICIAL / name
    assert hashlib.sha256(asset.read_bytes()).hexdigest() == record["sha256"], f"Official source changed: {name}"
print("Official repository:", source["repository"])
print("Pinned commit:", source["commit"])
print("Python:", sys.version.split()[0])
print("All bundled official files passed their SHA-256 checks.")


## Choose a CSV

Save your frozen possession CSV in the same `evaluation` folder as this notebook. Leave `CSV_FILENAME = ""` to automatically select the only CSV there, or type its name, for example `CSV_FILENAME = "my_team.csv"`. If several CSVs exist, enter the one you want. With no CSV present and the field blank, the notebook runs the official baseline demonstration.

`INCLUDE_OFFICIAL_BASELINE` adds the published baseline to the same board. `BOOTSTRAP_DRAWS = 1000` shows uncertainty on this small sample; set0 for a faster run without intervals. Every run is saved separately under ignored `local/evaluation/`; previous runs are preserved.


In [ ]:
# Type the name of a CSV saved beside this notebook, e.g. "my_team.csv".
# Leave blank to automatically pick the only CSV in this evaluation folder.
CSV_FILENAME = "Possession_Lens_017.csv"  # <-- YOUR CSV FILENAME HERE
INCLUDE_OFFICIAL_BASELINE = True
BOOTSTRAP_DRAWS = 1000

assert isinstance(BOOTSTRAP_DRAWS, int) and BOOTSTRAP_DRAWS >= 0
assert isinstance(CSV_FILENAME, str), "Enter the filename in quotes."
filename = CSV_FILENAME.strip()
available = sorted(p for p in EVALUATION.iterdir() if p.is_file() and p.suffix.lower() == ".csv")
print("CSV folder:", EVALUATION)
if available:
    print("Available CSVs:", ", ".join(p.name for p in available))
if filename:
    assert Path(filename).name == filename and filename.lower().endswith(".csv"), "Enter only the CSV filename, for example my_team.csv. Save it beside this notebook."
    selected_csv = EVALUATION / filename
    assert selected_csv.is_file(), f"Not found: {filename}. Save that file in {EVALUATION}, or correct CSV_FILENAME."
elif len(available) == 1:
    selected_csv = available[0]
elif len(available) > 1:
    raise ValueError("Several CSVs found. Enter one of these names in CSV_FILENAME: " + ", ".join(p.name for p in available))
else:
    selected_csv = None
print("Selected:", selected_csv.name if selected_csv else "Official baseline demonstration (no local CSV found)")


## Validate the exact file and run the official program

This cell invokes `validate_submission.py` and `run_evaluation.py` from the pinned official source, without replacing their scoring logic. It scores one frozen candidate plus the optional baseline. Format validation alone does not measure accuracy. If the official program rejects an entry, its message is shown and the notebook stops.


In [ ]:
run_folder = ROOT / "local/evaluation" / datetime.now().strftime("%Y%m%d-%H%M%S-%f")
inputs = run_folder / "results"
outputs = run_folder / "scores"
inputs.mkdir(parents=True, exist_ok=False)

baseline = OFFICIAL / "evaluation/results/baseline_smooth50.csv"
if selected_csv is None or INCLUDE_OFFICIAL_BASELINE:
    shutil.copy2(baseline, inputs / "baseline_smooth50.csv")
if selected_csv is not None:
    # Fixed display name avoids collisions with baseline names and unsafe filename characters.
    shutil.copy2(selected_csv, inputs / "frozen_candidate.csv")

for csv in sorted(inputs.glob("*.csv")):
    validated = subprocess.run([sys.executable, str(OFFICIAL / "evaluation/validate_submission.py"), str(csv)], cwd=ROOT, capture_output=True, text=True)
    print(validated.stdout)
    if validated.stderr:
        print(validated.stderr)
    assert validated.returncode == 0, f"Official validator rejected {csv.name}"

command = [sys.executable, str(OFFICIAL / "evaluation/run_evaluation.py"), "--results", str(inputs), "--gt", "public", "--out", str(outputs), "--bootstrap", str(BOOTSTRAP_DRAWS)]
completed = subprocess.run(command, cwd=ROOT, capture_output=True, text=True)
print(completed.stdout)
if completed.stderr:
    print(completed.stderr)
(run_folder / "official-output.txt").write_text(completed.stdout + completed.stderr)
assert completed.returncode == 0, "The official evaluator rejected an entry; see its message above."
run_record = {
    "upstream_commit": source["commit"],
    "reference": "official public five-minute sample",
    "input_sha256": {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in inputs.glob("*.csv")},
    "bootstrap_draws": BOOTSTRAP_DRAWS,
    "python": sys.version,
    "training_performed": False,
    "model_selection_performed": False,
}
(run_folder / "run-manifest.json").write_text(json.dumps(run_record, indent=2) + "\n")
print("Local results saved:", run_folder)


## Read the scores

**Spell F1** determines the ranking. **Macro F1** measures frame labels across H/A/D; **Dead F1** measures the dead-ball class. The confidence interval shows how uncertain spell F1 is on this short public sample. A high public score does not establish full-match performance.


In [ ]:
scoreboard = pd.read_csv(outputs / "scoreboard.csv")
columns = [c for c in ["rank", "submission", "status", "spell_f1_mean", "macro_f1", "f1_H", "f1_A", "f1_D", "spell_bootstrap_low", "spell_bootstrap_high", "n_gt_spells", "poss", "n_graded"] if c in scoreboard.columns]
display(scoreboard[columns].rename(columns={"spell_f1_mean": "Spell F1", "macro_f1": "Macro F1", "f1_H": "Home F1", "f1_A": "Away F1", "f1_D": "Dead F1", "poss": "Predicted possessions", "n_gt_spells": "Reference possessions", "n_graded": "Scored frames", "spell_bootstrap_low": "Spell CI low", "spell_bootstrap_high": "Spell CI high"}))
print("Files:")
display(FileLink(os.path.relpath(outputs / "scoreboard.csv", Path.cwd())))
display(FileLink(os.path.relpath(run_folder / "official-output.txt", Path.cwd())))
for report in sorted((outputs / "reports").glob("*.json")):
    display(FileLink(os.path.relpath(report, Path.cwd())))
print("Detailed official transition reports are also in", outputs / "reports")
print("Use aggregate scores for checking the frozen pipeline; do not inspect predictions against the challenge video or tune on public labels.")


## Reproduction notes

Source and SHA-256 hashes: `evaluation/official/SOURCE.json`. Licence: `evaluation/official/LICENSE`. Official files are unchanged. The public reference and published baseline come from the organisers’ public repository; the licensed match video/tracking are not included.

This notebook runs offline after installing the project's notebook dependencies. It uses the current Jupyter kernel for the official programs, so NumPy and pandas must be installed in that kernel. For a fresh environment, install `requirements-notebook.txt` from the project root. It never imports a training routine.

The default baseline should reproduce the published public spell F1 of about0.3256. The public sample must not select the final model; our model comparison and tuning remain on external matches.
